In [103]:
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets
import pandas as pd

db_api = DuckDBAPI()

df = pd.read_csv('/Users/katiefrields/170A-HW1/new_data/new_csrankings.csv', header = None)
df.columns = ['clean_name', 'full_name', 'first_name', 'surname', 'affiliation', 'homepage', 'scholar', 'middle_name']
df['index'] = df.index


In [154]:
df.shape

(30400, 9)

In [133]:
comparison_middle_name = {
    "output_column_name": "middle_name",
    "comparison_levels": [
        {
            "sql_condition": """
                middle_name_l IS NULL OR middle_name_l = '' OR
                middle_name_r IS NULL OR middle_name_r = ''
            """,
            "is_null_level": True,
            "label_for_charts": "missing"
        },
        {
            "sql_condition": """
                substr(middle_name_l, 1, 1) = substr(middle_name_r, 1, 1)
            """,
            "m_probability": .75,
            "u_probability": 0.15,
            "label_for_charts": "initials match"
        },
        {
            "sql_condition": """
                jaro_winkler_similarity(middle_name_l, middle_name_r) >= 0.85
            """,
            "m_probability": 0.75,
            "u_probability": 0.15,
            "label_for_charts": "fuzzy match"
        },
    ],
}

            
            
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        comparison_middle_name,
        cl.JaroWinklerAtThresholds("first_name", [0.9, 0.7]),
        cl.JaroWinklerAtThresholds("surname", [0.9, 0.10]),
        ],
        blocking_rules_to_generate_predictions=[
            block_on("surname", "affiliation")
        ],
        unique_id_column_name="index"
    )


linker = Linker(df, settings, db_api)

linker.training.estimate_probability_two_random_records_match(
    [block_on("surname", "affiliation")],
    recall=0.7,
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("surname", "affiliation")
)

#linker.training.estimate_parameters_using_expectation_maximisation(block_on("dob"))


pairwise_predictions = linker.inference.predict(threshold_match_weight=-10)

clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    pairwise_predictions, 0.95
)

df_clusters = clusters.as_pandas_dataframe()

Probability two random records match is estimated to be  4.22e-05.
This means that amongst all possible pairwise record comparisons, one in 23,714.74 are expected to match.  With 462,064,800 total possible comparisons, we expect a total of around 19,484.29 matching pairs
You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----
u probability not trained for middle_name - fuzzy match (comparison vector value: 0). This usually means the comparison level was never observed in the training data.

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).

----- Starting EM training session -----

Estimating t

In [132]:
pairwise_predictions.as_pandas_dataframe().nlargest(n=10, columns='match_probability')

,match_weight,match_probability,index_l,index_r,middle_name_l,middle_name_r,gamma_middle_name,first_name_l,first_name_r,gamma_first_name,surname_l,surname_r,gamma_surname,affiliation_l,affiliation_r,match_key
0,4.447693,0.956182,16386,16387,A.,Anup,1,Mansi,Mansi,3,Radke,Radke,3,VNIT Nagpur,VNIT Nagpur,0
4,4.447693,0.956182,16480,16481,P.,Parolin,1,Marcel,Marcel,3,Jackowski,Jackowski,3,USP,USP,0
5,4.447693,0.956182,16497,16499,F.,Fabiùn,1,Marcelo,Marcelo,3,Frias,Frias,3,University of Texas - El Paso,University of Texas - El Paso,0
6,4.447693,0.956182,16498,16499,Fabian,Fabiùn,1,Marcelo,Marcelo,3,Frias,Frias,3,University of Texas - El Paso,University of Texas - El Paso,0
7,4.447693,0.956182,16502,16503,G.,Garcia,1,Marcelo,Marcelo,3,Manzato,Manzato,3,USP-ICMC,USP-ICMC,0
8,4.447693,0.956182,16507,16509,K.,Keese,1,Marcelo,Marcelo,3,Albertini,Albertini,3,UFU,UFU,0
9,4.447693,0.956182,16515,16518,R.,Rita,1,Marcelo,Marcelo,3,Pias,Pias,3,Federal University of Rio Grande,Federal University of Rio Grande,0
10,4.447693,0.956182,16527,16528,H. C.,Helena Costa,1,Marcia,Marcia,3,Fampa,Fampa,3,UFRJ,UFRJ,0
11,4.447693,0.956182,16535,16536,\N,\N,1,Marcin,Marcin,3,Kaminski,Kaminski,3,University of Warsaw,University of Warsaw,0
12,4.447693,0.956182,16542,16547,A.,Antonio,1,Marco,Marco,3,Casanova,Casanova,3,PUC-RIO,PUC-RIO,0


In [87]:
sorted_df_clusters = df_clusters.groupby('cluster_id').head()

In [140]:
duplicate_value = df_clusters.duplicated(subset=['cluster_id'], keep = False)
duplicate_i = duplicate_value[duplicate_value == True].index.values

In [128]:
duplicate_i

array([   34,    35,    40, ..., 30165, 30170, 30264], shape=(1296,))

In [148]:
duplicate_table = df_clusters.iloc[duplicate_i].sort_values(by='cluster_id')
duplicate_table.head(n=30)

,cluster_id,clean_name,full_name,first_name,surname,affiliation,homepage,scholar,middle_name,index
31,31,A. P. Vinod,A. P. Vinod 0001,A.,Vinod,Nanyang Technological University,http://www.ntu.edu.sg/home/asvinod,NOSCHOLARPAGE\r,P.,31
34,31,A. Prasad Vinod,A. Prasad Vinod,A.,Vinod,Nanyang Technological University,http://www.ntu.edu.sg/home/asvinod,NOSCHOLARPAGE\r,Prasad,34
35,31,A. Prasad Vinod,A. Prasad Vinod 0001,A.,Vinod,Nanyang Technological University,http://www.ntu.edu.sg/home/asvinod,NOSCHOLARPAGE\r,Prasad,35
39,39,A. S. M. Hoque,A. S. M. Hoque,A.,Hoque,BUET,https://cse.buet.ac.bd/faculty_list/detail/asm...,3-Sb7tMAAAAJ\r,S. M.,39
40,39,A. S. M. Latiful Hoque,A. S. M. Latiful Hoque,A.,Hoque,BUET,https://cse.buet.ac.bd/faculty_list/detail/asm...,3-Sb7tMAAAAJ\r,S. M. Latiful,40
52,52,A. W. Roscoe,A. W. Roscoe 0001,A.,Roscoe,University of Oxford,http://www.cs.ox.ac.uk/bill.roscoe,miixtKcAAAAJ\r,W.,52
53,52,A. William Roscoe,A. William Roscoe,A.,Roscoe,University of Oxford,http://www.cs.ox.ac.uk/bill.roscoe,miixtKcAAAAJ\r,William,53
106,105,Abdeltawab M. Hendawi,Abdeltawab M. Hendawi,Abdeltawab,Hendawi,University of Rhode Island,https://homepage.cs.uri.edu/~ahendawi,ad3Gki4AAAAJ\r,M.,106
105,105,Abdeltawab M. A. Hendawi,Abdeltawab M. A. Hendawi,Abdeltawab,Hendawi,University of Rhode Island,https://homepage.cs.uri.edu/~ahendawi,ad3Gki4AAAAJ\r,M. A.,105
119,119,Abdulla K. Al-Ali,Abdulla K. Al-Ali,Abdulla,Al-Ali,Qatar University,http://www.qu.edu.qa/engineering/computer/facu...,s1LawbAAAAAJ\r,K.,119


In [156]:
clean_table = df_clusters.drop_duplicates(subset=['cluster_id'], keep='first')
clean_table.drop(columns=['cluster_id', 'index'], inplace=True)


/var/folders/dm/cwmd0_bn1lg6rxgh_wfkctpc0000gn/T/ipykernel_67778/2865588344.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_table.drop(columns=['cluster_id', 'index'], inplace=True)


In [157]:
clean_table.head()

,clean_name,full_name,first_name,surname,affiliation,homepage,scholar,middle_name
0,A Min Tjoa,A Min Tjoa,A,Tjoa,TU Wien,http://www.ifs.tuwien.ac.at/tjoa,x8qCMhcAAAAJ\r,Min
1,A. Akbari Azirani,A. Akbari Azirani,A.,Azirani,IUST,http://ce.iust.ac.ir/page.php?slct_pg_id=6537&...,pCil4_cAAAAJ\r,Akbari
2,A. Akbariazirani,A. Akbariazirani,A.,Akbariazirani,IUST,http://ce.iust.ac.ir/page.php?slct_pg_id=6537&...,pCil4_cAAAAJ\r,\N
3,A. Aldo Faisal,A. Aldo Faisal,A.,Faisal,Imperial College London,https://www.imperial.ac.uk/people/a.faisal,WjHjbrwAAAAJ\r,Aldo
4,A. Antony Franklin,A. Antony Franklin,A.,Franklin,IIT Hyderabad,http://www.iith.ac.in/~antony/index.html,LVfqLuoAAAAJ\r,Antony


In [158]:
clean_table.to_csv('clean_csrankings.csv', index = False, header = ['clean_name','full_name','first_name','surname','affiliation','homepage','scholar','middle_name'])


